<a href="https://colab.research.google.com/github/sterldb/Projet-miniville-M1-DJL/blob/lulu9/Mini_Jeux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from abc import ABC
import random as rd

class Affichage:
# Gere l'affichage des informations et les questions posées aux joueurs, aisi que leurs reponses
    def __init__(self, partie):
        self._partie = partie

    @property
    def partie(self):
        return self._partie

    def choixdes(self, joueur):
      # On cherche le monument 'Gare' par son nom dans la liste des monuments du joueur
      gare =[m for m in joueur.monuments if m.nom == "Gare"]

      # Si le joueur n'a pas de monument 'Gare' ou s'il n'est pas actif
      if not gare[0].actif:
          print(f"{joueur.nom} n'a pas construit la Gare. Lancement automatique de 1 dé.")
          return 1

      print(f"\n{joueur.nom}, vous avez construit la Gare !")
      while True:
          try:
              choix = int(input("Combien de dés voulez-vous lancer ? (1 ou 2) : "))
              if choix in [1, 2]:
                  return choix
              else:
                  print("Veuillez choisir 1 ou 2.")
          except ValueError:
              print("Veuillez entrer un nombre valide (1 ou 2).")

    def lancerdes(self, nb):
      #affiche et retourne le resultat du lancer de nb dés
      a = False
      if nb == 2:
          resultat1 = rd.randint(1, 6)
          resultat2 = rd.randint(1, 6)
          resultat = resultat1 + resultat2
          a = resultat1 == resultat2
      else:
          resultat = rd.randint(1, 6)

      print(f"Résultat du lancer de {nb} dé(s) : {resultat}")
      return resultat, a

    def choixrelance(self, joueur):
      # On cherche le monument 'Tour radio' par son nom dans la liste des monuments du joueur
      tour_radio = [m for m in joueur.monuments if m.nom == "Tour radio"]

      # Si le joueur n'a pas la Tour radio ou si elle n'est pas active, pas de relance possible
      if not tour_radio[0].actif:
          return False

      print(f"\n{joueur.nom}, vous avez construit la Tour radio ! Vous pouvez relancer vos dés.")
      while True:
          choix = input("Voulez-vous relancer vos dés ? (oui/non) : ")

          # Étape 1 : On teste si l'utilisateur a écrit un nombre
          est_un_nombre = False
          try:
              float(choix)   #si c'est un nombre, on rentre dans une saisie invalide et si c'est un autre mots que "oui","non" ça lève une ValueError puis on verifie la validité du mots entré dans l'Etape 2.
              est_un_nombre = True
          except ValueError:
              pass

          if est_un_nombre:
              print("Saisie invalide (les nombres ne sont pas acceptés). Veuillez répondre par 'oui' ou 'non'.")
              continue

          # Étape 2 : Si ce n'est pas un nombre, on vérifie si la réponse est valide
          if choix == "oui":
              return True
          elif choix == "non":
              return False
          else:
              print("Veuillez répondre par 'oui' ou 'non'.")

    def afficherplateau(self):
      #affiche le plateau dans son etat actuel
      #parcourir les joueurs
      for i in self.partie.liste_joueurs:
        print(i.nom, ' : ')
        #parcourir les batiments , création d'une liste sans doublons
        aff = []
        nb = []
        for j in i.batiments:
          for k in aff:
            if j.nom == k:
              nb[aff.index(k)] += 1
          if j.nom not in aff:
            aff.append(j.nom)
            nb.append(1)
        for j in range(len(aff)):
          print(aff[j], ' x', nb[j])
        print("Solde : ", i.solde)
        if i.monuments[0].actif:
          print("Gare construite")
        else:
          print("Gare non construite")
        if i.monuments[1].actif:
          print("Centre commercial construit")
        else:
          print("Centre commercial non construit")
        if i.monuments[2].actif:
          print("Parc d'attraction construit")
        else:
          print("Parc d'attraction non construit")
        if  i.monuments[3].actif:
          print("Tour radio construit")
        else:
          print("Tour radio non construit")
        print()
      for i in self.partie.batiments_banque:
        print(i.nom, ' : ', i.cout)



    def choixtour(self):
      #affiche et retourne le choix du joueur actif de contruire un batiment ou de passer cette phase
      print(self.partie.joueur_actif.nom, " voulez vous construire un batiment ? (oui/non)")
      while true:
        a = input()
        if a == "oui":
          return True
        if a == "non":
          return False
        else :
          print("Veuillez répondre par oui ou non")

    def batimentsachetables(self):
      #affiche  et return sous forme de liste les batiments achetables par le joueur
      joueur = self.partie.joueur_actif
      achetables = []

      # Liste de tuples contenant les coûts de chaque bâtiment
      couts_batiments = [
          ("Champs de blé", 1),
          ("Ferme", 1),
          ("Boulangerie", 1),
          ("Café", 2),
          ("Supérette", 2),
          ("Forêt", 3),
          ("Fromagerie", 5),
          ("Fabrique de meubles", 3),
          ("Mine", 6),
          ("Centre d'affaires", 8),
          ("Stade", 6),
          ("Chaîne de télévision", 7),
          ("Restaurant", 3),
          ("Verger", 3),
          ("Marché de fruits et légumes", 2)
      ]
      couts_monuments=[("Gare",4),
          ("Parc d'attractions",16),
          ("Tour radio",22),
          ("Centre commercial",10)]

      print(f"\nBâtiments disponibles à la banque et achetables par {joueur.nom} (Solde : {joueur.solde} pièces) :")

      # On parcourt les bâtiments de la banque stockés sous la forme [(nom_batiment, exemplaire_disponible)]
      for nom_bat, quantite in self.partie.batiments_banque:
          if quantite > 0:
              # Recherche du coût correspondant au bâtiment dans notre liste de tuples
              cout = 999
              for nom_recherche, prix in couts_batiments:
                  if nom_recherche == nom_bat:
                      cout = prix
                      break
              #Si le batiment est disponible on regarde si le joueur a les moyens de l'obtenir.
              if joueur.solde >= cout:
                  # Les bâtiments violets uniques ne peuvent être possédés qu'en un seul exemplaire
                  if nom_bat in ["Centre d'affaires", "Stade", "Chaîne de télévision"]:
                      deja_possede = False
                      for b in joueur.batiments:
                          if b.nom == nom_bat:
                              deja_possede = True
                              break
                      if deja_possede:
                          continue

                  achetables.append(nom_bat)

      for j in joueur.monuments:
        if not j.actif:
          for nom,prix in couts_monuments:
            if nom==j.nom and joueur.solde>=prix:
              achetables.append(nom)
              break


      if not achetables:
          print("Aucun bâtiment n'est achetable.")  # Si la liste est vide.

      return achetables

    def choixjoueur(self):
      #affiche et retoure la choix du joueur victime du vol
      joueur_actif = self.partie.joueur_actif

      # On filtre la liste des joueurs pour exclure le joueur actif
      adversaires = [j for j in self.partie.liste_joueurs if j is not self.partie.joueur_actif]


      print("Choisissez un joueur parmi les suivants :")
      for numero, adversaire in enumerate(adversaires):
          print(f"{numero + 1} - {adversaire.nom}\n")
      # Tant que l'utilisatuer n'a pas saisie un joueur correct(qui est associe a un numero) on redemande et on capte l'erreur ValueError si l'utilisateur ecrit un mots

      while True:
          try:
              choix = int(input("Entrez le numéro du joueur cible : "))
              if 1 <= choix <= len(adversaires):
                  return adversaires[choix - 1]
              else:
                  print("Numéro invalide. Veuillez réessayer.")
          except ValueError:
              print("Veuillez entrer un nombre valide.")

    def echangebatiment(self, joueur1, joueur2):
      # Affiche et retourne le choix d'echange de batiments entre la liste des batiments du joueur 1 et celle du joueur 2, le joueur 1 choisit
      # Cette fonction retourne le nom des batiments echangés sous la forme d'un tuple (nom_bat_joueur1 (batiments pris à joueur2), nom_bat_joueur2 (batiment donné pour le joueur1))

      print(f"\nÉchange de bâtiments entre {joueur1.nom} et {joueur2.nom} ")

      # On filtre les bâtiments pour exclure complètement les bâtiments de type "radio"
      offre_possibles = [i for i in joueur1.batiments if i.types != "radio"]
      demande_possibles = [j for j in joueur2.batiments if j.types != "radio"]

      # Affichage global initial des deux listes pour la stratégie.
      print(f"\n{joueur1.nom}, voici tes bâtiments :")
      for num, bat in enumerate(offre_possibles):
          print(f"  {num + 1} - {bat.nom} (Type: {bat.types})\n")

      print(f"\nBâtiments échangeables chez {joueur2.nom} :")
      for num, bat in enumerate(demande_possibles):
          print(f"  {num + 1} - {bat.nom} (Type: {bat.types})\n")

      # Choix du bâtiment à offrir.
      print(f"\n{joueur1.nom}, choisissez le numéro du bâtiment de votre liste à donner :")
      while True:
          try:
              choix1 = int(input("Numéro : "))
              if 1 <= choix1 <= len(offre_possibles):
                  batiment_offert = offre_possibles[choix1 - 1]
                  break
              else:
                  print("Numéro invalide. Veuillez réessayer.")
          except ValueError:
              print("Veuillez entrer un nombre valide.")

      # Choix du bâtiment à prendre.
      print(f"\n{joueur1.nom}, choisissez le numéro du bâtiment à prendre chez {joueur2.nom} :")
      while True:
          try:
              choix2 = int(input("Numéro : "))
              if 1 <= choix2 <= len(demande_possibles):
                  batiment_demande = demande_possibles[choix2 - 1]
                  break
              else:
                  print("Numéro invalide. Veuillez réessayer.")
          except ValueError:
              print("Veuillez entrer un nombre valide.")

      return (batiment_demande.nom, batiment_offert.nom)
      # On mets bcp le signe \n pour que l'utilisateur trouve le l'affichage aéré.
    def Nombrejoueurs(self):
      #demande et retourne le nombre de joueurs dans la partie, de 2 a 4
      print("A combien de joueurs voulez vous jouer ?")
      a = input()
      if a in ["2", "3", "4"]:
        return int(a)
      else:
        print("Veuillez répondre par 2, 3 ou 4")
        return self.Nombrejoueurs()
    def nomjoueur(self, i):
      print("joueur", i, "veuillez entrer votre nom")
      a = input()
      return a

    def construction(self,gratuit:bool):
      joueur=self.partie.joueur_actif
      bat_dispo=self.batimentsachetables()

      if not bat_dispo:
        print(f"Vous ne pouvez pas construire de batiment.")
        return None

      print("\n --- Voici les batiments que vous pouvez construire!---")
      for num, bat in enumerate(bat_dispo):
          print(f"\n  {num + 1} - {bat} ")

      if self.choixtour():
        while True:
          try:
              choix = int(input("Numéro : "))
              if 1 <= choix <= len(bat_dispo):
                  self.partie.joueur.construire(bat_dispo[choix-1],gratuit)
                  break
              else:
                  print("Numéro invalide. Veuillez réessayer.")
          except ValueError:
              print("Veuillez entrer un nombre valide.")
      else:
        print(f"{joueur.nom} n'a pas voulue construire de batiment.")
        return None



    def partie_gagnée(self, joueur):
      print(f"{joueur.nom} a gagné la partie !")

class Partie:
#Gere Une partie et le cours du jeu, commencer a jouer avec la methode initialisation partie
    def __init__(
        self, affichage = None,compteur_tours: int =0 , liste_joueurs:list =[] , batiments_banque: list =[], ordre_tour: list=[], joueur_actif: Joueur = None
    ):
        self._affichage=affichage
        self._compteur_tours = compteur_tours
        self._liste_joueurs = liste_joueurs
        self._batiments_banque = batiments_banque
        self._ordre_tour = ordre_tour
        self._joueur_actif = joueur_actif
        self.initialisation()


    @property
    def affichage(self):
      return self._affichage

    @affichage.setter
    def affichage(self, value):
      self._affichage = value

    @property
    def compteur_tours(self):
        return self._compteur_tours

    @property
    def liste_joueurs(self):
        return self._liste_joueurs

    @liste_joueurs.setter
    def liste_joueurs(self,joueur):
      self._liste_joueurs.append(joueur)


    @property
    def batiments_banque(self):
        return self._batiments_banque

    @batiments_banque.setter
    def batiments_banque(self, value):
        self._batiments_banque = value

    @property
    def ordre_tour(self):
        return self._ordre_tour

    @ordre_tour.setter
    def ordre_tour(self, value):
        self._ordre_tour = value

    @property
    def joueur_actif(self):
        return self._joueur_actif

    @joueur_actif.setter
    def joueur_actif(self, value):
        self._joueur_actif = value

    def tour(self):
      self.joueur_actif = self.ordre_tour[0]
      del self.ordre_tour[0]
      self.ordre_tour.append(self.joueur_actif)
      resultatdes = self.affichage.lancerdes(self.affichage.choixdes(self.joueur_actif))
      for i in self.ordre_tour:
        for j in i.batiments:
          j.activation(resultatdes[0])

      self.affichage.construction(False)
      a = 1
      for i in self.joueur_actif.monuments:
        a = a * i.actif
      if a :
        self.affichage.partie_gagnée(joueur_actif)
      if self.joueur_actif.monuments[2].actif and resultatdes[1]:
        for i in range(len(self.ordre_tour)-1):
          self.ordre_tour[-(1+i)] == self.ordre_tour[-(2+i)]
        self.ordre_tour[0] = self.joueur_actif
      self.tour()

    def initialisation(self):
    #initialise la partie
    self.affichage = Affichage(self)
    nbjoueurs = self.affichage.Nombrejoueurs()
    self.batiments_banque = [("Champs de blé", 6),
          ("Ferme", 6),
          ("Boulangerie", 6),
          ("Café", 6),
          ("Supérette", 6),
          ("Forêt", 6),
          ("Fromagerie", 6),
          ("Fabrique de meubles", 6),
          ("Mine", 6),
          ("Centre d'affaires", 4),
          ("Stade", 4),
          ("Chaîne de télévision", 4),
          ("Restaurant", 6),
          ("Verger", 6),
          ("Marché de fruits et légumes", 6)]
    for i in range(nbjoueurs):
      self.liste_joueurs.append(Joueur(self, nomjoueur(self, i)))
    self.ordre_tour = self.liste_joueurs
    self.affichage.afficherplateau()
    self.tour()




class Joueur:
#represente un joueur et ses ressources
    def __init__(self,  nom, partie, monuments = [], batiments = [], liste_types = [],solde: int = 3,):
        self._solde = solde
        self._monuments = [Monuments("Gare", 4, self), Monuments("Centre commercial", 10, self), Monuments("Parc d'attractions", 16, self), Monuments("Tour radio", 22, self)]
        #liste des monuments, au debut sont tous inactif, ATTENTION ILS SONT DANS L'ORDRE (Gare, Centre commercial, Parc d'attraction, Tour radio)
        #on vérifiera si un monument est construit en vérifiant la condition de son ordre (pour gare on verifiera if Joueur.monuments[0]:)
        self._liste_types =  [["blé",0],["vache",0],["boite de carton",0],["cafe",0],["engrenage",0],["radio",0],["usine",0],["ballon",0]]
        self._batiments = [Ferme(self), Boulangerie(self)]
        self._partie = partie
        #liste du nombre de cartes de chaque "type" (petit emoticone) que le joueur possede (sert pour calculer les fromageries et les fabriques de meubles et les marchés de fruits et legumes plus ptet autre chose)
        #DANS L'ORDRE (consequament l'endroit dans la liste): blé 0, vache 1, boite en carton 2, tasse de café 3, engrenage 4, radio 5, usine 6, ballon 7
        #La liste des types sera donc donner sous la forme [("blé",0),....,("ballon",0)]
        self._nom = nom

    @property
    def solde(self):
        return self._solde

    @solde.setter
    def solde(self, value):
        self._solde = value

    @property
    def nom(self):
      return self._nom

    @property
    def monuments(self):
        return self._monuments

    @property
    def batiments(self):
        return self._batiments

    @batiments.setter
    def batiments(self, value):
        self._batiments = value

    @property
    def partie(self):
        return self._partie

    @property
    def liste_types(self):
        return self._liste_types

    @liste_types.setter
    def liste_types(self, value):
        self._liste_types = value

    def construire(self,Batiment:str, gratuit:bool):
      #on apelle cette fonction en donnant le NOM du batiment, il faut donc que cette methone reconaisse si c'est un batiment ou un monument puis le construire
      if Batiment in ["Gare", "Centre commercial", "Parc d'attractions", "Tour radio"]:# si c'est un monument
        for Bb in self._monuments:
          if Bb.nom() == Batiment:
                  self._solde -= Bb.cout
                    Bb[0].actif() = True


      if Batiment in ["Champs de blé", "Ferme", "Boulangerie", "Café", "Supérette", "Forêt", "Centre d'affaires", "Stade", "Chaîne de télévision", "Fromagerie", "Fabrique de meubles", "Mine", "Restaurant", "Verger", "Marché de fruits et légumes"]:
        for Bb in self._partie.batiments_banque():# on recherhce notre batiment dans la liste des batiment en banque de la partie en cours
              if gratuit == True : # on regarde si il s'agit d'une construction gratuite ou pas
                  i = self._partie.batiment_banque().index(Bb)
                  self._partie.batiment_banque([i]) = (Bb[0],Bb[1]-1) # on rédéfinit le nombre d'exmplaire de la carte parmis les carte en banque
                  self._batiments.append(Bb[0])
              if gratuit == False:
                  self._solde = self._solde - Bb.cout() # on redéfinit le nouveau sole du joueur après son achat
                  i = self._partie.batiment_banque().index(Bb)
                  self._partie.batiment_banque([i]) = (Bb[0],Bb[1]-1) # on rédéfinit le nombre d'exmplaire de la carte parmis les carte en banque
                  self._batiments.append(Bb[0])






    def payer(self, joueur, argent):
      # méthode pour qu'un joueur paye un autre joueur
      if self.solde == 0:
        pass
      else:
        if self.solde < argent:
          # On donne tout ce qu'il reste au joueur, pas plus !
          joueur.solde += self.solde
          self.solde = 0
        else:
          joueur.solde += argent
          self.solde -= argent


class Batiments(ABC):
#classe abstraite servant de base pour les batiments classiques.
    def __init__(self, nom: str, cout: int, type_batiment: str, proprietaire: Joueur, nombre:int):
        self._nom = nom
        self._cout = cout
        self._type = type_batiment
        self._proprietaire = proprietaire
        self._nombre = nombre

    @property
    def nom(self):
        return self._nom

    @property
    def cout(self):
        return self._cout

    @property
    def types(self):
        return self._type

    @property
    def proprietaire(self):
        return self._proprietaire

    @proprietaire.setter
    def proprietaire(self, value):
        self._proprietaire = value

    @property
    def nombre(self):
        return self._nombre

class Champs_de_ble(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Champs de blé",1,"blé",proprietaire,1)
        self._propriétaire.liste_types[0][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 1, le propriétaire gagne 1 pièce de la banque
        if resultat_des == self._nombre:
            self.proprietaire.solde += 1

class Ferme(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Ferme",1,"vache",proprietaire,2)
        self._propriétaire.liste_types[1][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 2, le propriétaire gagne 1 pièce de la banque
        if resultat_des == self._nombre:
            self.proprietaire.solde += 1

class Boulangerie(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Boulangerie",1,"boite de carton",proprietaire,[2,3])
        self._propriétaire.liste_types[2][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 2 ou 3, et que le propriétaire est le joueur actif
        if resultat_des in self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
            self.proprietaire.solde += 1

class Cafe(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Café",2,"cafe",proprietaire,3)
        self._propriétaire.liste_types[3][1] += 1
    def activation(self,resultat_des):
        # Si le résultat du dé est 3, le propriétaire gagne 1 pièce de la part du joueur qui a lancés les dés.
        if resultat_des == self._nombre:
            self.proprietaire.partie.joueur_actif.payer(self._proprietaire,1)

class Superette(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Supérette",2,"boite de carton",proprietaire,4)
        self._propriétaire.liste_types[2][1] += 1
    def activation(self,resultat_des):
        # Si le résultat du dé est 4, et que le propriétaire est le joueur actif
        if resultat_des in self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
            self.proprietaire.solde += 3

class Foret(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Forêt",3,"engrenage",proprietaire,5)
        self._propriétaire.liste_types[4][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 5, le propriétaire gagne 1 pièce de la banque
        if resultat_des == self._nombre:
            self.proprietaire.solde += 1

class Fromagerie(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Fromagerie",5,"usine",proprietaire,7)
        self._propriétaire.liste_types[6][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 7, et que le propriétaire est le joueur actif
        if resultat_des == self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
            self.proprietaire.solde += 3*self._proprietaire.liste_types[1]

class Fabrique_de_meubles(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Fabrique de meubles",3,"usine",proprietaire,8)
        self._propriétaire.liste_types[6][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 7, et que le propriétaire est le joueur actif
        if resultat_des in self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
          self.proprietaire.solde += 3*self._proprietaire.liste_types[4]


class Mine(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Forêt",6,"engrenage",proprietaire,9)
        self._propriétaire.liste_types[4][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 9, le propriétaire gagne 5 pièce de la banque
        if resultat_des == self._nombre:
          self.proprietaire.solde += 5


class Centre_affaires(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Centre d'affaires",8,"radio",proprietaire,6)
        self._propriétaire.liste_types[5][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 6, et que le propriétaire est le joueur actif
        if resultat_des == self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
          victime =  self.proprietaire.partie.affichage.choixjoueur()
          A = self.proprietaire.partie.affichage.echangebatiment(self.proprietaire, victime)
          if A is not None:
              self.proprietaire.construire(A[0],True) #on construit le batiments de l'échange gratuitement (car c'est un echange) Maintenant il faut retirer le batiments chez victime
              for i in victime.batiments:
                if A[0]==i.nom:
                  for y in victime.liste_types:
                    if y[0]==i.types:
                      y[1]-=1
                  victime.batiments.remove(i)
                  break
              victime.construire(A[1], True)
              for i in self.proprietaire.batiments:
                if A[1]==i.nom:
                  for y in self.proprietaire.liste_types:
                    if y[0]==i.types:
                      y[1]-=1
                  self.proprietaire.batiments.remove(i)
                  break

class Stade(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Stade",6,"radio",proprietaire,6)
        self._propriétaire.liste_types[5][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 6, et que le propriétaire est le joueur actif
        if resultat_des == self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
          for i in self.prorietaire.partie.liste_joueurs:
            if i is not self.proprietaire:
              i.payer(self.proprietaire,2)


class Chaine_Television(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Chaîne de télévision",7,"radio",proprietaire,6)
        self._propriétaire.liste_types[5][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 6, et que le propriétaire est le joueur actif
        if resultat_des == self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
          victime=self.proprietaire.partie.affichage.choixjoueur()
          victime.payer(self.proprietaire,5)


class Restaurant(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Restaurant",3,"cafe",proprietaire,[9, 10])
        self._propriétaire.liste_types[3][1] += 1
    def activation(self,resultat_des):
        # Si le résultat du dé est 9 ou 10, le propriétaire gagne 2 pièce de la part du joueur qui a lancés les dés.
        if resultat_des in self._nombre:
            self.proprietaire.partie.joueur_actif.payer(self._proprietaire,1)

class Verger(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Verger",3,"blé",proprietaire,10)
        self._propriétaire.liste_types[0][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 10, le propriétaire gagne 1 pièce de la banque
        if resultat_des == self._nombre:
            self.proprietaire.solde += 3

class Marche_de_fruits_et_legumes(Batiments):
    def __init__(self, proprietaire):

        super().__init__("Marché de fuits et legumes",2,"ballon",proprietaire,[11, 12])
        self._propriétaire.liste_types[7][1] += 1

    def activation(self,resultat_des):
        # Si le résultat du dé est 11 ou 12, et que le propriétaire est le joueur actif
        if resultat_des in self._nombre and self.proprietaire == self.proprietaire.partie.joueur_actif:
            self.proprietaire.solde += 2*self.proprietaire.liste_types[0]

class Monuments:
#classe pour les Monuments
    def __init__(self, nom: str, cout:int, proprietaire: Joueur, actif : bool = False):
        self._nom = nom
        self._actif = actif
        self._cout = cout
        self._proprietaire = proprietaire

    @property
    def nom(self):
        return self._nom

    @property
    def actif(self):
        return self._actif

    @actif.setter
    def actif(self, value):
        self._actif = value

    @property
    def cout(self):
        return self._cout

    @property
    def proprietaire(self):
        return self._proprietaire